# k07 — Frozen results package (Stage 6)
Assembles the authoritative numerical tables from the frozen k05 (evaluation + statistics) and k06 (recording-level audit) outputs. **No metric is recomputed**: every value is copied from a named source file, and `T0_provenance.tsv` records the SHA-256 of each source and each emitted table.
T1 master per-cell PR-AUC · T2 attack-family PR-AUC (tokens with ≥50 positive windows, both definitions) · T3 hypotheses (exact enumeration for B cells and strata; random bootstrap flagged where used) · T4 operating points · T5 recording-level audit of the B cells with leave-one-out · T6 seed vs recording variability.

In [ ]:
import os, glob
os.makedirs('/kaggle/working/code', exist_ok=True)
open('/kaggle/working/code/exp_package.py','w').write('"""Stage 6 — frozen results package. Reads ONLY the frozen k05 (stats+eval meta) and k06 (audit) outputs and emits the\nauthoritative tables. No metric is recomputed here except formatting; every number is copied from a named source file.\nUsage: python exp_package.py --stats <k05 stats dir> --eval <k05 eval dir> --audit <k06 audit dir> --out /kaggle/working/package"""\nimport os, json, glob, hashlib, argparse\nMODELS = [\'Rule\', \'LightGBM\', \'LightGBM_S\', \'DeepSets\', \'GRU\', \'GraphSAGE\', \'GraphSAGE_rewired\']\nDISP = {\'Rule\': \'Rule\', \'LightGBM\': \'LightGBM\', \'LightGBM_S\': \'LightGBM+S\', \'DeepSets\': \'DeepSets\', \'GRU\': \'GRU\',\n        \'GraphSAGE\': \'GraphSAGE\', \'GraphSAGE_rewired\': \'GraphSAGE-rewired\'}\nROLE = {\'test_01\': \'A known/known\', \'test_02\': \'B unknown vehicle/known attack\', \'test_03\': \'C known vehicle/unknown attack\', \'test_04\': \'D unknown/unknown\'}\nH_LABEL = {\'H1\': \'H1 GraphSAGE - DeepSets\', \'H1a\': \'H1a GraphSAGE - GraphSAGE-rewired\', \'H2\': \'H2 GraphSAGE - GRU\',\n           \'H3\': \'H3 GraphSAGE - LightGBM+S\', \'ladder_LGBS_minus_LGB\': \'LightGBM+S - LightGBM\'}\nAUD_KEY = {\'H1\': \'H1_GS_minus_DeepSets\', \'H1a\': \'H1a_GS_minus_rewired\', \'H2\': \'H2_GS_minus_GRU\', \'H3\': \'H3_GS_minus_LGBS\',\n           \'ladder_LGBS_minus_LGB\': \'ladder_LGBS_minus_LGB\'}\n\ndef sha(p):\n    h = hashlib.sha256()\n    with open(p, \'rb\') as f:\n        for b in iter(lambda: f.read(1 << 20), b\'\'): h.update(b)\n    return h.hexdigest()\n\ndef f4(x): return f\'{x:.4f}\'\ndef pm(x): return f\'{x:+.4f}\'\n\ndef main(STATS, EVAL, AUDIT, OUT):\n    os.makedirs(OUT, exist_ok=True)\n    R = json.load(open(os.path.join(STATS, \'results.json\')))\n    A = json.load(open(os.path.join(AUDIT, \'audit.json\')))\n    T = {}\n    # T1 master per-cell PR-AUC\n    L = [\'cell\\trole\\trecordings\\twindows\\tpositives\\tprevalence\\thours\\t\' + \'\\t\'.join(DISP[m] + \' PR-AUC (mean+-SD over 5 seeds)\' for m in MODELS)]\n    for k, c in R[\'per_cell\'].items():\n        L.append(\'\\t\'.join([k, ROLE[k.split(\'/\')[1]], str(c[\'n_files\']), str(c[\'windows\']), str(c[\'pos\']),\n                            f4(c[\'prevalence_chance_ap\']), f\'{c["hours"]:.3f}\'] +\n                           [f"{c[\'models\'][m][\'ap_mean\']:.4f}+-{c[\'models\'][m][\'ap_sd\']:.4f}" for m in MODELS]))\n    T[\'T1_master_cells.tsv\'] = L\n    # T2 attack-family table (tokens with >=50 positive windows in the cell)\n    L = [\'cell\\trole\\ttoken\\tpositive_windows\\t\' + \'\\t\'.join(DISP[m] for m in MODELS) + \'\\tdefinition\']\n    for k, c in R[\'per_cell\'].items():\n        toks = sorted({t for m in MODELS for t in c[\'models\'][m][\'per_token\']})\n        for t in toks:\n            for defn, field in ((\'token recordings only\', \'ap_token_recordings_mean\'), (\'token positives vs all cell negatives\', \'ap_token_pos_vs_all_cell_neg_mean\')):\n                row = [k, ROLE[k.split(\'/\')[1]].split()[0], t, str(c[\'models\'][MODELS[0]][\'per_token\'][t][\'pos\'])]\n                row += [f4(c[\'models\'][m][\'per_token\'][t][field]) for m in MODELS]\n                L.append(\'\\t\'.join(row + [defn]))\n    T[\'T2_attack_family.tsv\'] = L\n    # T3 hypothesis table: exact enumeration (k06) for B cells and strata; random bootstrap (k05) for non-B cells\n    L = [\'contrast\\tscope\\tdifference\\tCI95_low\\tCI95_high\\tCI90_low\\tCI90_high\\tdecision\\tp\\tmethod\\tn_resamples\']\n    for h, label in H_LABEL.items():\n        ak = AUD_KEY[h]\n        for sname, v in A[\'summary\'].get(ak, {}).items():\n            L.append(\'\\t\'.join([label, sname, pm(v[\'point_full_sample\']), pm(v[\'ci95\'][0]), pm(v[\'ci95\'][1]), pm(v[\'ci90\'][0]), pm(v[\'ci90\'][1]),\n                                v[\'decision\'], f"{v[\'p_exact_two_sided\']:.4f}", \'exact per-cell enumeration (k06)\', str(v[\'n_distinct_resamples\'])]))\n        for k, c in A[\'cells\'].items():\n            v = c[\'exact_bootstrap\'][ak]\n            L.append(\'\\t\'.join([label, k, pm(v[\'point_full_sample\']), pm(v[\'ci95\'][0]), pm(v[\'ci95\'][1]), pm(v[\'ci90\'][0]), pm(v[\'ci90\'][1]),\n                                v[\'decision\'], f"{v[\'p_exact_two_sided\']:.4f}", \'exact enumeration (k06)\', str(v[\'n_distinct_resamples\'])]))\n        for k, c in R[\'per_cell\'].items():\n            if k.endswith(\'test_02\'): continue\n            v = c[\'contrasts\'][h if h in c[\'contrasts\'] else h]\n            L.append(\'\\t\'.join([label, k + \' (secondary cell)\', pm(v[\'diff_point\']), pm(v[\'ci95\'][0]), pm(v[\'ci95\'][1]), pm(v[\'ci90\'][0]), pm(v[\'ci90\'][1]),\n                                v[\'decision\'], f"{v[\'p_boot_two_sided\']:.4f}", \'random bootstrap 2000 (k05)\', \'2000\']))\n    L.append(\'\\t\'.join([\'Holm-adjusted p (family: H1a/H2/H3 B-summary + H1/H1a/H2/H3 per B cell; H1 B-summary is primary and uncorrected)\'] + [\'\'] * 10))\n    for m, p in sorted(R[\'holm_family\'][\'p_holm\'].items(), key=lambda kv: kv[1]):\n        L.append(\'\\t\'.join([m, \'Holm (on k05 random-bootstrap p)\', \'\', \'\', \'\', \'\', \'\', \'\', f\'{p:.4f}\', \'k05\', \'\']))\n    T[\'T3_hypotheses.tsv\'] = L\n    # T4 operating points\n    L = [\'cell\\trole\\thours\\tmodel\\tMacroF1_at_maxF1_threshold\\trecall_at_1FAh\\tachieved_FAh_at_1FAh\\tallowed_FP_1FAh\\trecall_at_5FAh\\tachieved_FAh_at_5FAh\\tallowed_FP_5FAh\']\n    for k, c in R[\'per_cell\'].items():\n        for m in MODELS:\n            v = c[\'models\'][m]\n            L.append(\'\\t\'.join([k, ROLE[k.split(\'/\')[1]].split()[0], f\'{c["hours"]:.3f}\', DISP[m], f4(v[\'macro_f1_mean\']),\n                                f4(v[\'recall_at_1FAh_mean\']), f\'{v["achieved_FAh_at_1FAh_mean"]:.2f}\', str(v[\'allowed_fp_test_1FAh\']),\n                                f4(v[\'recall_at_5FAh_mean\']), f\'{v["achieved_FAh_at_5FAh_mean"]:.2f}\', str(v[\'allowed_fp_test_5FAh\'])]))\n    T[\'T4_operating_points.tsv\'] = L\n    # T5 recording-level audit of the B cells\n    L = [\'cell\\trecording\\ttoken\\twindows\\tpositives\\tshare_of_cell_positives\\tprevalence\\t\' + \'\\t\'.join(DISP[m] for m in MODELS)\n         + \'\\tLOO_H1_without_this_recording\\tLOO_H1_change\\tLOO_H1a_without\\tLOO_H1a_change\']\n    for k, c in A[\'cells\'].items():\n        for f in c[\'files\']:\n            lo = c[\'leave_one_out\'].get(f[\'file\'], {})\n            row = [k, f[\'file\'], f[\'token\'], str(f[\'windows\']), str(f[\'pos\']), f4(f[\'share_of_cell_positives\']), f4(f[\'prevalence\'])]\n            row += [f4(f[\'ap_mean\'][m]) for m in MODELS]\n            row += ([pm(lo[\'contrasts\'][\'H1_GS_minus_DeepSets\']), pm(lo[\'delta_vs_full\'][\'H1_GS_minus_DeepSets\']),\n                     pm(lo[\'contrasts\'][\'H1a_GS_minus_rewired\']), pm(lo[\'delta_vs_full\'][\'H1a_GS_minus_rewired\'])] if lo else [\'\', \'\', \'\', \'\'])\n            L.append(\'\\t\'.join(row))\n    T[\'T5_recording_audit_B.tsv\'] = L\n    # T6 variability\n    L = [\'cell\\tmodel\\tSD_across_5_seeds\\tSD_across_recording_resamples\']\n    for k, c in A[\'cells\'].items():\n        for m in MODELS:\n            v = c[\'seed_vs_recording_variability\'][m]\n            L.append(\'\\t\'.join([k, DISP[m], f4(v[\'sd_across_seeds\']), f4(v[\'sd_across_recording_resamples\'])]))\n    T[\'T6_variability.tsv\'] = L\n    for name, lines in T.items():\n        open(os.path.join(OUT, name), \'w\').write(\'\\n\'.join(lines) + \'\\n\')\n    # provenance\n    src = {\'k05_results.json\': os.path.join(STATS, \'results.json\'), \'k06_audit.json\': os.path.join(AUDIT, \'audit.json\')}\n    for p in sorted(glob.glob(os.path.join(EVAL, \'set_*\', \'*_meta.json\'))) + sorted(glob.glob(os.path.join(EVAL, \'set_*\', \'id_permutation_test.json\'))):\n        src[\'/\'.join(p.split(os.sep)[-2:])] = p\n    prov = [\'file\\tsha256\\tbytes\']\n    for nm, p in src.items(): prov.append(f\'{nm}\\t{sha(p)}\\t{os.path.getsize(p)}\')\n    for name in T: prov.append(f\'{name}\\t{sha(os.path.join(OUT, name))}\\t{os.path.getsize(os.path.join(OUT, name))}\')\n    open(os.path.join(OUT, \'T0_provenance.tsv\'), \'w\').write(\'\\n\'.join(prov) + \'\\n\')\n    for nm in [\'T0_provenance.tsv\'] + list(T):\n        print(\'\\n########\', nm)\n        print(open(os.path.join(OUT, nm)).read())\n\nif __name__ == \'__main__\':\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--stats\', required=True); ap.add_argument(\'--eval\', required=True); ap.add_argument(\'--audit\', required=True)\n    ap.add_argument(\'--out\', default=\'/kaggle/working/package\'); a = ap.parse_args()\n    main(a.stats, a.eval, a.audit, a.out)\n')
ST = glob.glob('/kaggle/input/**/stats/results.json', recursive=True); AU = glob.glob('/kaggle/input/**/audit/audit.json', recursive=True)
EV = glob.glob('/kaggle/input/**/eval/set_01/test_01_meta.json', recursive=True)
print(ST, AU, EV); assert len(ST) == 1 and len(AU) == 1 and len(EV) == 1
STATS = os.path.dirname(ST[0]); AUDIT = os.path.dirname(AU[0]); EVAL = os.path.dirname(os.path.dirname(EV[0]))
print(STATS, AUDIT, EVAL)


In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, '/kaggle/working/code/exp_package.py', '--stats', STATS, '--eval', EVAL, '--audit', AUDIT, '--out', '/kaggle/working/package'])
print('exit', r.returncode)
